In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets, Features, Image, Value
from PIL import Image as PILImage
import random, pandas as pd
from collections import defaultdict

from datasets import load_dataset
ds = load_dataset("SimulaMet-HOST/Kvasir-VQA")["raw"]

In [ ]:
from datasets import concatenate_datasets

# Remove invalid question entries
valid_ds = ds.filter(lambda ex: ex["question"] and ex["question"] != "none")

# Identify abnormal samples (source != 'normal')
abnormal_ids = set(
    ex["img_id"] for ex in valid_ds if ex["source"].lower() != "normal"
)

# add "Does this image contain any finding?" = "yes" for abnormal cases
seen_ids = set()
added_examples = []
for ex in valid_ds:
    if ex["img_id"] in abnormal_ids and ex["img_id"] not in seen_ids:
        added_examples.append({
            "image": ex["image"],
            "source": ex["source"],
            "question": "Does this image contain any finding?",
            "answer": "yes",
            "img_id": ex["img_id"]
        })
        seen_ids.add(ex["img_id"])

# Combine cleaned data with added questions
modified_ds = Dataset.from_list(added_examples)
cleaned_ds = concatenate_datasets([valid_ds, modified_ds])


In [ ]:
print('cleaned size:', len(cleaned_ds))

cleaned size: 62747


In [ ]:
import random
from datasets import load_dataset, DatasetDict

# all unique img_ids
all_ids = sorted(set(cleaned_ds["img_id"]))

# Shuffle & split those IDs into 80/10/10
random.seed(42)
random.shuffle(all_ids)

n = len(all_ids)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
# n_test will be whatever is left
train_ids = set(all_ids[:n_train])
val_ids   = set(all_ids[n_train : n_train + n_val])
test_ids  = set(all_ids[n_train + n_val :])



In [ ]:
def filter_by_id(example, id_set):
    return example["img_id"] in id_set



In [ ]:
train_ds = cleaned_ds.filter(lambda ex: filter_by_id(ex, train_ids),
                     batched=False)
val_ds   = cleaned_ds.filter(lambda ex: filter_by_id(ex, val_ids),
                     batched=False)
test_ds  = cleaned_ds.filter(lambda ex: filter_by_id(ex, test_ids),
                     batched=False)

In [ ]:
dataset = DatasetDict({
    "train":      train_ds,
    "validation": val_ds,
    "test":       test_ds,
})

print({k: len(v) for k, v in dataset.items()})

{'train': 50105, 'validation': 6287, 'test': 6355}


In [ ]:
# keeping copy for tracking ( image,question,answer,source,img_id)  #dataset_full['test'][i]['img_id'] (or ['source'])
dataset_full = dataset

data = dataset_full.remove_columns(["source", "img_id"])
data

DatasetDict({
    train: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 50105
    })
    validation: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6287
    })
    test: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6355
    })
})

In [ ]:
print('SPLIT SIZES:', {k: len(v) for k, v in data.items()})

SPLIT SIZES: {'train': 50105, 'validation': 6287, 'test': 6355}


In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoProcessor, BitsAndBytesConfig, Idefics3ForConditionalGeneration

USE_LORA = True
USE_8BIT = True

processor = AutoProcessor.from_pretrained(
    "HuggingFaceM4/Idefics3-8B-Llama3",
    do_image_splitting=False
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules='.*(text_model|modality_projection|perceiver_resampler).*(down_proj|gate_proj|up_proj|k_proj|q_proj|v_proj|o_proj).*$',
    use_dora=True,
    init_lora_weights="gaussian"
)

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False
)


model = Idefics3ForConditionalGeneration.from_pretrained(
    "HuggingFaceM4/Idefics3-8B-Llama3",
    quantization_config=bnb_config,
    device_map={"": "cuda:0"},
)

model.add_adapter(lora_config)
model.enable_adapters()


[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.
Loading weights: 100%|██████████| 729/729 [00:12<00:00, 56.35it/s] 


In [ ]:
class MyDataCollator:
    def __init__(self, processor):
        self.processor = processor
        self.image_token_id = processor.tokenizer.convert_tokens_to_ids("<image>")

    def __call__(self, examples):
        texts, images = [], []
        for example in examples:
            image = example["image"].convert("RGB")
            question = example["question"]
            answer = example["answer"]
            messages = [
                {"role": "user", "content": [
                    {"type": "text", "text": "Answer as a medical specialist"},
                    {"type": "image"},
                    {"type": "text", "text": question}
                ]},
                {"role": "assistant", "content": [
                    {"type": "text", "text": answer}
                ]}
            ]
            text = processor.apply_chat_template(messages, add_generation_prompt=False)
            texts.append(text.strip())
            images.append([image])
        batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
        labels = batch["input_ids"].clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        labels[labels == self.image_token_id] = -100
        batch["labels"] = labels
        return batch

data_collator = MyDataCollator(processor)

In [ ]:

import torch


_img_proc = processor.image_processor
_MEAN = torch.tensor(_img_proc.image_mean).view(1, 1, 3, 1, 1)
_STD  = torch.tensor(_img_proc.image_std).view(1, 1, 3, 1, 1)

def _to_unit(pv):

    return pv * _STD.to(pv.device, pv.dtype) + _MEAN.to(pv.device, pv.dtype)

def _to_norm(unit):

    return (unit - _MEAN.to(unit.device, unit.dtype)) / _STD.to(unit.device, unit.dtype)


def pgd_perturb_batch(model, batch, eps=8/255, alpha=2/255, num_iter=7, random_start=True):

    was_training = model.training
    model.eval()  

    pv = batch["pixel_values"].detach()
    unit_clean = _to_unit(pv).detach()
    other = {k: v for k, v in batch.items() if k != "pixel_values"}

    if num_iter <= 0:
        if was_training: model.train()
        return pv

    delta = (torch.empty_like(unit_clean).uniform_(-eps, eps)
             if random_start else torch.zeros_like(unit_clean)).to(unit_clean.dtype)

    for _ in range(num_iter):
        delta.requires_grad_(True)
        unit_adv = torch.clamp(unit_clean + delta, 0.0, 1.0)
        out = model(pixel_values=_to_norm(unit_adv), **other)
        grad = torch.autograd.grad(out.loss, delta)[0]
        with torch.no_grad():
            delta = delta + alpha * grad.sign()               
            delta = torch.clamp(delta, -eps, eps)
            delta = torch.clamp(unit_clean + delta, 0.0, 1.0) - unit_clean
        delta = delta.detach()

    unit_adv = torch.clamp(unit_clean + delta, 0.0, 1.0).detach()
    if was_training:
        model.train()
    return _to_norm(unit_adv).detach()


def fgsm_perturb_batch(model, batch, eps=8/255):
    return pgd_perturb_batch(model, batch, eps=eps, alpha=eps, num_iter=1, random_start=False)

In [ ]:

import random as _random
import torch
from transformers import Trainer


ADV_ATTACK   = "pgd"     
ADV_EPS      = 8/255
ADV_ALPHA    = 2/255
ADV_NUM_ITER = 7          
ADV_MIX      = True      
ADV_PROB     = 1.0        

class AdversarialTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):

        if not (model.training and torch.is_grad_enabled()):
            outputs = model(**inputs)
            loss = outputs.loss
            return (loss, outputs) if return_outputs else loss


        do_adv = _random.random() < ADV_PROB
        attack = ADV_ATTACK
        if attack == "mixed":
            attack = _random.choice(["pgd", "fgsm"])

        if not do_adv:
            outputs = model(**inputs)
            loss = outputs.loss
            return (loss, outputs) if return_outputs else loss


        if attack == "fgsm":
            adv_pv = fgsm_perturb_batch(model, inputs, eps=ADV_EPS)
        else:
            adv_pv = pgd_perturb_batch(model, inputs, eps=ADV_EPS,
                                       alpha=ADV_ALPHA, num_iter=ADV_NUM_ITER)


        adv_inputs = dict(inputs)
        adv_inputs["pixel_values"] = adv_pv.to(inputs["pixel_values"].dtype)
        adv_outputs = model(**adv_inputs)
        adv_loss = adv_outputs.loss

        if ADV_MIX:
            clean_outputs = model(**inputs)
            loss = 0.5 * clean_outputs.loss + 0.5 * adv_loss
            outputs = adv_outputs
        else:
            loss = adv_loss
            outputs = adv_outputs

        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=False,   
    warmup_steps=100,
    learning_rate=1e-4,
    lr_scheduler_type="constant",
    weight_decay=0.01,
    output_dir=r"idefics3_checkpoint_lora_adv_training_pgd_fgsm",
    optim="paged_adamw_8bit",
    save_total_limit=1,
    bf16=True,
    remove_unused_columns=False,
    report_to="none",
    logging_steps=200,
    eval_steps=200,
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=400,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
)

trainer = AdversarialTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=data["train"],
    eval_dataset=data["validation"],
)

print(f"Adversarial training: attack={ADV_ATTACK} eps={ADV_EPS:.4f} "
      f"alpha={ADV_ALPHA:.4f} iters={ADV_NUM_ITER} mix={ADV_MIX} prob={ADV_PROB}")
trainer.train()

In [ ]:
save_dir = "idefics3_lora_adv_training_pgd_fgsm_final"
trainer.save_model(save_dir)
processor.save_pretrained(save_dir)
print("saved to", save_dir)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.85it/s]


saved to idefics3_lora_adv_training_pgd_fgsm_final


## Evaluation

In [ ]:
import evaluate
import torch
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import Levenshtein

def clean_answer(text):
    if "Assistant:" in text:
        return text.split("Assistant:")[-1].strip()
    return text.strip()

In [ ]:
preds, refs = [], []
model.eval()

for ex in tqdm(data['test']):
    image = ex["image"].convert("RGB")
    question = ex["question"]
    answer = ex["answer"]

    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Answer as a medical specialist"},
            {"type": "image"},
            {"type": "text", "text": question}
        ]
    }]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[[image]], return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=64)
    pred = clean_answer(processor.batch_decode(output_ids, skip_special_tokens=True)[0])

    preds.append(pred)
    refs.append(answer if isinstance(answer, list) else [answer])

  0%|          | 0/6355 [00:00<?, ?it/s]/home/rifat/anaconda3/envs/vlm_attack/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/rifat/anaconda3/envs/vlm_attack/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for

In [ ]:
refs_single = [r[0] for r in refs]
accuracy = sum(p in r for p, r in zip(preds, refs)) / len(refs) * 100

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

bleu_res   = bleu.compute(predictions=preds, references=[[r] for r in refs_single])
rouge_res  = rouge.compute(predictions=preds, references=refs_single)
meteor_res = meteor.compute(predictions=preds, references=refs_single)

j_scores = []
for r, p in zip(refs_single, preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard = sum(j_scores) / len(j_scores) * 100

vectorizer = TfidfVectorizer().fit(refs_single + preds)
cosine = cosine_similarity(vectorizer.transform(refs_single),
                           vectorizer.transform(preds)).diagonal().mean() * 100

def normalized_levenshtein(s1, s2):
    if not s1 and not s2: return 0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def similarity_score(a_ij, o_q_i, tau=0.5):
    nl = normalized_levenshtein(a_ij, o_q_i)
    return 1 - nl if nl < tau else 0

def average_levenshtein_similarity(ground_truth, predicted):
    total = 0
    for refs_, pred in zip(ground_truth, predicted):
        if not pred: continue
        total += max(similarity_score(ref, pred) for ref in refs_)
    return total / len(ground_truth) * 100

levenshtein_score = average_levenshtein_similarity(refs, preds)

[nltk_data] Downloading package wordnet to /home/rifat/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/rifat/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/rifat/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
results = {
    "accuracy (%)": round(accuracy, 2),
    "bleu": bleu_res,
    "rouge": rouge_res,
    "meteor": meteor_res,
    "jaccard (%)": round(jaccard, 2),
    "cosine (%)": round(cosine, 2),
    "levenshtein (%)": round(levenshtein_score, 2)
}
for k, v in results.items():
    print(f"{k}:\n{v}\n")

accuracy (%):
85.52

bleu:
{'bleu': 0.7584352861829691, 'precisions': [0.8560999473169263, 0.7638488170802077, 0.7132286590868074, 0.7094363791631084], 'brevity_penalty': 1.0, 'length_ratio': 1.0152823412546803, 'translation_length': 13287, 'reference_length': 13087}

rouge:
{'rouge1': 0.9065077092600363, 'rouge2': 0.16335715158254138, 'rougeL': 0.9048375299936493, 'rougeLsum': 0.904880123351604}

meteor:
{'meteor': 0.5190461181265038}

jaccard (%):
88.24

cosine (%):
75.99

levenshtein (%):
89.45



## Adversarial Robustness Analysis

In [ ]:
import random

synonyms_dict = {
    "type": ["kind", "category"],
    "procedure": ["process", "test"],
    "image": ["picture", "visual"],
    "abnormalities": ["irregularities", "issues"],
    "present": ["visible", "detected"],
    "easy": ["simple", "straightforward"],
    "detect": ["identify", "locate"],
    "polyp": ["lesion", "mass", "growth"],
    "size": ["dimension", "measurement"],
    "instrument": ["tool", "equipment"],
    "removed": ["extracted", "taken out"],
    "where": ["in what part", "in which area", "location of"],
    "how many": ["number of", "count of", "total"]
}

insert_words = [
    "possibly", "likely", "evidently", "apparently", "visibly",
    "clinically", "endoscopically", "approximately", "typically"
]



def synonym_replacement(question, synonyms_dict):
    words = question.split()
    new_words = [random.choice(synonyms_dict.get(w.lower(), [w])) for w in words]
    return " ".join(new_words)

def random_insertion(question, insert_words):
    words = question.split()
    if not words: return question
    insert_word = random.choice(insert_words)
    insert_pos = random.randint(0, len(words))
    return " ".join(words[:insert_pos] + [insert_word] + words[insert_pos:])

def random_deletion(question, p=0.2):
    words = question.split()
    if len(words) == 1: return question
    return " ".join([w for w in words if random.random() > p])

def print_ground_truth_answers(img_id, dataset):
    entries = [e for e in dataset if e['img_id'] == img_id]
    if not entries:
        print("Image ID not found in dataset.")
        return
    print(f"Ground Truth for Image ID: {img_id}\n")
    for i, entry in enumerate(entries):
        print(f"Q{i+1}: {entry['question']}")
        print(f"A{i+1}: {entry['answer']}\n")



In [ ]:
attacked_preds = []
for ex in tqdm(data['test']):
    image = ex["image"]
    question = ex["question"]
    answer = ex["answer"]

    # only synonum replacement
    attacked_q = synonym_replacement(question, synonyms_dict)

    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Answer as a medical specialist"},
            {"type": "image"},
            {"type": "text", "text": attacked_q}
        ]
    }]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[[image]], return_tensors="pt", padding=True).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=64)

    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)
    attacked_preds.append(pred)

100%|██████████| 6355/6355 [4:24:47<00:00,  2.50s/it]  


In [ ]:
# Acc@Attack: Percentage of correct predictions under attack
acc_at_attack = sum(p in r for p, r in zip(attacked_preds, refs)) / len(refs) * 100

# ASR: Attack success rate (when prediction changes)
asr = sum(p1 != p2 for p1, p2 in zip(preds, attacked_preds)) / len(preds) * 100

print(f"Acc@Attack (%): {acc_at_attack:.3f}")
print(f"Attack Success Rate (ASR %): {asr:.3f}")

Acc@Attack (%): 83.714
Attack Success Rate (ASR %): 4.044


In [ ]:
# Prepare flat reference list
refs_single = [r[0] for r in refs]

# Accuracy (same as accuracy under attack)
adv_accuracy = sum(p in r for p, r in zip(attacked_preds, refs)) / len(refs) * 100

# Evaluate metrics
bleu_res_adv   = bleu.compute(predictions=attacked_preds, references=[[r] for r in refs_single])
rouge_res_adv  = rouge.compute(predictions=attacked_preds, references=refs_single)
meteor_res_adv = meteor.compute(predictions=attacked_preds, references=refs_single)

# Jaccard Similarity
j_scores_adv = []
for r, p in zip(refs_single, attacked_preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores_adv.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard_adv = sum(j_scores_adv) / len(j_scores_adv) * 100

# Cosine Similarity (TF-IDF)
vectorizer_adv = TfidfVectorizer().fit(refs_single + attacked_preds)
ref_vecs_adv  = vectorizer_adv.transform(refs_single)
pred_vecs_adv = vectorizer_adv.transform(attacked_preds)
cos_sims_adv  = cosine_similarity(ref_vecs_adv, pred_vecs_adv).diagonal()
cosine_adv = cos_sims_adv.mean() * 100

# Levenshtein Similarity
def normalized_levenshtein(s1, s2):
    if not s1 and not s2:
        return 0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def similarity_score(a_ij, o_q_i, tau=0.5):
    nl = normalized_levenshtein(a_ij, o_q_i)
    return 1 - nl if nl < tau else 0

def average_levenshtein_similarity(ground_truth, predicted):
    total_score = 0
    for refs, pred in zip(ground_truth, predicted):
        if not pred:
            continue
        max_score = max(similarity_score(ref, pred) for ref in refs)
        total_score += max_score
    return total_score / len(ground_truth) * 100

levenshtein_adv = average_levenshtein_similarity(refs, attacked_preds)

adv_results = {
    "accuracy (%)": round(adv_accuracy, 2),
    "bleu": bleu_res_adv,
    "rouge": rouge_res_adv,
    "meteor": meteor_res_adv,
    "jaccard (%)": round(jaccard_adv, 2),
    "cosine (%)": round(cosine_adv, 2),
    "levenshtein (%)": round(levenshtein_adv, 2)
}

for k, v in adv_results.items():
    print(f"{k}:\n{v}\n")


accuracy (%):
83.71

bleu:
{'bleu': 0.7665008271051733, 'precisions': [0.8526478546579049, 0.7829787234042553, 0.7348784624081401, 0.7374439461883409], 'brevity_penalty': 0.9883177110804165, 'length_ratio': 0.9883854206464431, 'translation_length': 12935, 'reference_length': 13087}

rouge:
{'rouge1': 0.8809627045290853, 'rouge2': 0.15346353270445026, 'rougeL': 0.879182275057443, 'rougeLsum': 0.8788519628964869}

meteor:
{'meteor': 0.5021102308288278}

jaccard (%):
85.94

cosine (%):
73.48

levenshtein (%):
86.6



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def visualize_text_perturbations(data, preds_clean, preds_perturbed, perturb_fn, num_samples=5):
    indices = np.random.choice(len(data), size=num_samples, replace=False)

    for idx in indices:
        sample = data[int(idx)]
        image = sample["image"]
        question = sample["question"]
        answer = sample["answer"]

        perturbed_question = perturb_fn(question)

        # Show image with text overlay
        plt.figure(figsize=(10, 5))
        plt.imshow(image)
        plt.axis("off")
        plt.title("Text Perturbation Example", fontsize=12)

        # Display original and perturbed content
        full_text = (
            f"Ground Truth: {answer}\n"
            f"Original Q: {question}\n"
            f"Original A: {preds_clean[int(idx)]}\n"
            f"Perturbed Q: {perturbed_question}\n"
            f"Perturbed A: {preds_perturbed[int(idx)]}"
        )
        plt.figtext(0.5, -0.13, full_text, wrap=True, ha="center", fontsize=9)
        plt.tight_layout()
        plt.show()

In [ ]:
visualize_text_perturbations(
    data=data['test'],
    preds_clean=preds,
    preds_perturbed=attacked_preds,
    perturb_fn=lambda q: synonym_replacement(q, synonyms_dict),
    num_samples=5
)

### with all 3 text perturbations

In [ ]:
attacked_preds = []
for ex in tqdm(data['test']):
    image = ex["image"]
    question = ex["question"]
    answer = ex["answer"]

    # Combined
    attacked_q = synonym_replacement(question, synonyms_dict)
    attacked_q = random_insertion(attacked_q, insert_words)
    attacked_q = random_deletion(attacked_q)

    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Answer as a medical specialist"},
            {"type": "image"},
            {"type": "text", "text": attacked_q}
        ]
    }]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[[image]], return_tensors="pt", padding=True).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=64)

    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)
    attacked_preds.append(pred)

  0%|          | 0/6355 [00:00<?, ?it/s]/home/rifat/anaconda3/envs/vlm_attack/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/rifat/anaconda3/envs/vlm_attack/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
100%|██████████| 6355/6355 [4:27:52<00:00,  2.53s/it]   


In [ ]:
# Acc@Attack: Percentage of correct predictions under attack
acc_at_attack = sum(p in r for p, r in zip(attacked_preds, refs)) / len(refs) * 100

# ASR: Attack success rate (when prediction changes)
asr = sum(p1 != p2 for p1, p2 in zip(preds, attacked_preds)) / len(preds) * 100

print(f"Acc@Attack (%): {acc_at_attack:.3f}")
print(f"Attack Success Rate (ASR %): {asr:.3f}")

Acc@Attack (%): 73.942
Attack Success Rate (ASR %): 16.884


In [ ]:
# Prepare flat reference list
refs_single = [r[0] for r in refs]

# Accuracy (same as accuracy under attack)
adv_accuracy = sum(p in r for p, r in zip(attacked_preds, refs)) / len(refs) * 100

# Evaluate metrics
bleu_res_adv   = bleu.compute(predictions=attacked_preds, references=[[r] for r in refs_single])
rouge_res_adv  = rouge.compute(predictions=attacked_preds, references=refs_single)
meteor_res_adv = meteor.compute(predictions=attacked_preds, references=refs_single)

# Jaccard Similarity
j_scores_adv = []
for r, p in zip(refs_single, attacked_preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores_adv.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard_adv = sum(j_scores_adv) / len(j_scores_adv) * 100

# Cosine Similarity (TF-IDF)
vectorizer_adv = TfidfVectorizer().fit(refs_single + attacked_preds)
ref_vecs_adv  = vectorizer_adv.transform(refs_single)
pred_vecs_adv = vectorizer_adv.transform(attacked_preds)
cos_sims_adv  = cosine_similarity(ref_vecs_adv, pred_vecs_adv).diagonal()
cosine_adv = cos_sims_adv.mean() * 100

# Levenshtein Similarity
def normalized_levenshtein(s1, s2):
    if not s1 and not s2:
        return 0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def similarity_score(a_ij, o_q_i, tau=0.5):
    nl = normalized_levenshtein(a_ij, o_q_i)
    return 1 - nl if nl < tau else 0

def average_levenshtein_similarity(ground_truth, predicted):
    total_score = 0
    for refs, pred in zip(ground_truth, predicted):
        if not pred:
            continue
        max_score = max(similarity_score(ref, pred) for ref in refs)
        total_score += max_score
    return total_score / len(ground_truth) * 100

levenshtein_adv = average_levenshtein_similarity(refs, attacked_preds)

adv_results = {
    "accuracy (%)": round(adv_accuracy, 2),
    "bleu": bleu_res_adv,
    "rouge": rouge_res_adv,
    "meteor": meteor_res_adv,
    "jaccard (%)": round(jaccard_adv, 2),
    "cosine (%)": round(cosine_adv, 2),
    "levenshtein (%)": round(levenshtein_adv, 2)
}

for k, v in adv_results.items():
    print(f"{k}:\n{v}\n")


accuracy (%):
73.94

bleu:
{'bleu': 0.6759187857309435, 'precisions': [0.751321940378573, 0.6879294890947116, 0.642973708068903, 0.63543874523103], 'brevity_penalty': 0.9970921355077524, 'length_ratio': 0.9970963551616108, 'translation_length': 13049, 'reference_length': 13087}

rouge:
{'rouge1': 0.7778225523141808, 'rouge2': 0.12938143828739695, 'rougeL': 0.7760138329441004, 'rougeLsum': 0.7758677142203896}

meteor:
{'meteor': 0.43923936664972496}

jaccard (%):
75.86

cosine (%):
66.88

levenshtein (%):
76.53



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def visualize_text_perturbations(data, preds_clean, preds_perturbed, perturb_fn, num_samples=5):
    indices = np.random.choice(len(data), size=num_samples, replace=False)

    for idx in indices:
        sample = data[int(idx)]
        image = sample["image"]
        question = sample["question"]
        answer = sample["answer"]

        perturbed_question = perturb_fn(question)

        # Show image with text overlay
        plt.figure(figsize=(10, 5))
        plt.imshow(image)
        plt.axis("off")
        plt.title("Text Perturbation", fontsize=12)

        # Display original and perturbed content
        full_text = (
            # f"Ground Truth: {answer}\n"
            f"Original Q: {question}\n"
            f"Original A: {preds_clean[int(idx)]}\n"
            f"Perturbed Q: {perturbed_question}\n"
            f"Perturbed A: {preds_perturbed[int(idx)]}"
        )
        plt.figtext(0.5, -0.12, full_text, wrap=True, ha="center", fontsize=9)
        plt.tight_layout()
        plt.show()

In [ ]:
#to apply multiple func

def full_perturbation(q):
    q = synonym_replacement(q, synonyms_dict)
    q = random_insertion(q, insert_words)
    q = random_deletion(q)
    return q

visualize_text_perturbations(
    data=data['test'],
    preds_clean=preds,
    preds_perturbed=attacked_preds,
    perturb_fn=full_perturbation,
    num_samples=5
)

In [ ]:
### image

In [ ]:
from PIL import Image, ImageFilter, ImageEnhance
import torchvision.transforms as T
import numpy as np
import io

# Gaussian noise
def add_gaussian_noise(img, mean=0, std=10):
    np_img = np.array(img).astype(np.float32)
    noise = np.random.normal(mean, std, np_img.shape)
    noisy_img = np.clip(np_img + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy_img)

# Gaussian blur
def apply_blur(img, radius=2):
    return img.filter(ImageFilter.GaussianBlur(radius))

# Brightness shift
def shift_brightness(img, factor=1.5):  # >1 brightens, <1 darkens
    enhancer = ImageEnhance.Brightness(img)
    return enhancer.enhance(factor)

# JPEG compression
def jpeg_compress(img, quality=30):
    buffer = io.BytesIO()
    img.save(buffer, format='JPEG', quality=quality)
    return Image.open(buffer)

In [ ]:
# Image perturbation evaluation
image_attacked_preds = []

for ex in tqdm(data['test']):
    image = ex["image"]
    question = ex["question"]
    answer = ex["answer"]

    # Apply one or more image perturbation
    perturbed_image = add_gaussian_noise(image)
    perturbed_image = apply_blur(perturbed_image)
    perturbed_image = shift_brightness(perturbed_image, factor=0.7)
 

    # Prompt
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Answer as a medical specialist"},
            {"type": "image"},
            {"type": "text", "text": question}
        ]
    }]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[[perturbed_image]], return_tensors="pt", padding=True).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=64)

    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)
    image_attacked_preds.append(pred)



100%|██████████| 6355/6355 [4:40:46<00:00,  2.65s/it]  


In [ ]:
# Accuracy under image perturbation
acc_at_attack_img = sum(p in r for p, r in zip(image_attacked_preds, refs)) / len(refs) * 100

# Attack success rate
asr_img = sum(p1 != p2 for p1, p2 in zip(preds, image_attacked_preds)) / len(preds) * 100

print(f"Image Acc@Attack (%): {acc_at_attack_img:.2f}")
print(f"Image ASR (%): {asr_img:.2f}")

Image Acc@Attack (%): 82.60
Image ASR (%): 8.43


In [ ]:
# Prepare flat reference list
refs_single = [r[0] for r in refs]

# Accuracy (same as accuracy under attack)
adv_accuracy = sum(p in r for p, r in zip(image_attacked_preds, refs)) / len(refs) * 100

# Evaluate metrics
bleu_res_adv   = bleu.compute(predictions=image_attacked_preds, references=[[r] for r in refs_single])
rouge_res_adv  = rouge.compute(predictions=image_attacked_preds, references=refs_single)
meteor_res_adv = meteor.compute(predictions=image_attacked_preds, references=refs_single)

# Jaccard Similarity
j_scores_adv = []
for r, p in zip(refs_single, image_attacked_preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores_adv.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard_adv = sum(j_scores_adv) / len(j_scores_adv) * 100

# Cosine Similarity (TF-IDF)
vectorizer_adv = TfidfVectorizer().fit(refs_single + image_attacked_preds)
ref_vecs_adv  = vectorizer_adv.transform(refs_single)
pred_vecs_adv = vectorizer_adv.transform(image_attacked_preds)
cos_sims_adv  = cosine_similarity(ref_vecs_adv, pred_vecs_adv).diagonal()
cosine_adv = cos_sims_adv.mean() * 100

# Levenshtein Similarity
def normalized_levenshtein(s1, s2):
    if not s1 and not s2:
        return 0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def similarity_score(a_ij, o_q_i, tau=0.5):
    nl = normalized_levenshtein(a_ij, o_q_i)
    return 1 - nl if nl < tau else 0

def average_levenshtein_similarity(ground_truth, predicted):
    total_score = 0
    for refs, pred in zip(ground_truth, predicted):
        if not pred:
            continue
        max_score = max(similarity_score(ref, pred) for ref in refs)
        total_score += max_score
    return total_score / len(ground_truth) * 100

levenshtein_adv = average_levenshtein_similarity(refs, image_attacked_preds)

adv_results = {
    "accuracy (%)": round(adv_accuracy, 2),
    "bleu": bleu_res_adv,
    "rouge": rouge_res_adv,
    "meteor": meteor_res_adv,
    "jaccard (%)": round(jaccard_adv, 2),
    "cosine (%)": round(cosine_adv, 2),
    "levenshtein (%)": round(levenshtein_adv, 2)
}

for k, v in adv_results.items():
    print(f"{k}:\n{v}\n")

accuracy (%):
82.6

bleu:
{'bleu': 0.7128669802298857, 'precisions': [0.8141349906055788, 0.7243084324468796, 0.6703910614525139, 0.6532596263820053], 'brevity_penalty': 1.0, 'length_ratio': 1.0573851914113241, 'translation_length': 13838, 'reference_length': 13087}

rouge:
{'rouge1': 0.8733806489824344, 'rouge2': 0.16092957718234696, 'rougeL': 0.8718236904892573, 'rougeLsum': 0.8718764767369358}

meteor:
{'meteor': 0.5027916547977457}

jaccard (%):
85.04

cosine (%):
73.73

levenshtein (%):
85.94



### lowering image quality

In [ ]:
# Image perturbation evaluation
image_attacked_preds = []

for ex in tqdm(data['test']):
    image = ex["image"]
    question = ex["question"]
    answer = ex["answer"]

    # lowering image quality

    perturbed_image = jpeg_compress(image, quality=25)

    # Prompt
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Answer as a medical specialist"},
            {"type": "image"},
            {"type": "text", "text": question}
        ]
    }]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[[perturbed_image]], return_tensors="pt", padding=True).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=64)

    pred = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
    pred = clean_answer(pred)
    image_attacked_preds.append(pred)


100%|██████████| 6355/6355 [4:35:59<00:00,  2.61s/it]  


In [ ]:
# Accuracy under image perturbation
acc_at_attack_img = sum(p in r for p, r in zip(image_attacked_preds, refs)) / len(refs) * 100

# Attack success rate
asr_img = sum(p1 != p2 for p1, p2 in zip(preds, image_attacked_preds)) / len(preds) * 100

print(f"Image Acc@Attack (%): {acc_at_attack_img:.2f}")
print(f"Image ASR (%): {asr_img:.2f}")

Image Acc@Attack (%): 73.60
Image ASR (%): 21.23


In [ ]:
# Prepare flat reference list
refs_single = [r[0] for r in refs]

# Accuracy (same as accuracy under attack)
adv_accuracy = sum(p in r for p, r in zip(image_attacked_preds, refs)) / len(refs) * 100

# Evaluate metrics
bleu_res_adv   = bleu.compute(predictions=image_attacked_preds, references=[[r] for r in refs_single])
rouge_res_adv  = rouge.compute(predictions=image_attacked_preds, references=refs_single)
meteor_res_adv = meteor.compute(predictions=image_attacked_preds, references=refs_single)

# Jaccard Similarity
j_scores_adv = []
for r, p in zip(refs_single, image_attacked_preds):
    set_r, set_p = set(r.split()), set(p.split())
    j_scores_adv.append(len(set_r & set_p) / len(set_r | set_p) if (set_r | set_p) else 0)
jaccard_adv = sum(j_scores_adv) / len(j_scores_adv) * 100

# Cosine Similarity (TF-IDF)
vectorizer_adv = TfidfVectorizer().fit(refs_single + image_attacked_preds)
ref_vecs_adv  = vectorizer_adv.transform(refs_single)
pred_vecs_adv = vectorizer_adv.transform(image_attacked_preds)
cos_sims_adv  = cosine_similarity(ref_vecs_adv, pred_vecs_adv).diagonal()
cosine_adv = cos_sims_adv.mean() * 100

# Levenshtein Similarity
def normalized_levenshtein(s1, s2):
    if not s1 and not s2:
        return 0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def similarity_score(a_ij, o_q_i, tau=0.5):
    nl = normalized_levenshtein(a_ij, o_q_i)
    return 1 - nl if nl < tau else 0

def average_levenshtein_similarity(ground_truth, predicted):
    total_score = 0
    for refs, pred in zip(ground_truth, predicted):
        if not pred:
            continue
        max_score = max(similarity_score(ref, pred) for ref in refs)
        total_score += max_score
    return total_score / len(ground_truth) * 100

levenshtein_adv = average_levenshtein_similarity(refs, image_attacked_preds)

adv_results = {
    "accuracy (%)": round(adv_accuracy, 2),
    "bleu": bleu_res_adv,
    "rouge": rouge_res_adv,
    "meteor": meteor_res_adv,
    "jaccard (%)": round(jaccard_adv, 2),
    "cosine (%)": round(cosine_adv, 2),
    "levenshtein (%)": round(levenshtein_adv, 2)
}

for k, v in adv_results.items():
    print(f"{k}:\n{v}\n")

accuracy (%):
73.6

bleu:
{'bleu': 0.6645021718980412, 'precisions': [0.7424003441353599, 0.6897142104569999, 0.6316039228092376, 0.6028835884656462], 'brevity_penalty': 1.0, 'length_ratio': 1.0657904791013983, 'translation_length': 13948, 'reference_length': 13087}

rouge:
{'rouge1': 0.7738269316572319, 'rouge2': 0.13960443312170295, 'rougeL': 0.7719416741331994, 'rougeLsum': 0.7714774912112143}

meteor:
{'meteor': 0.44717502943125464}

jaccard (%):
75.59

cosine (%):
66.13

levenshtein (%):
75.7



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_perturbed_samples(data, preds_clean, preds_perturbed, perturb_fn, num_samples=5):
    indices = np.random.choice(len(data), size=num_samples, replace=False)

    for idx in indices:
        sample = data[int(idx)]
        image = sample["image"]
        question = sample["question"]
        answer = sample["answer"]

        perturbed_image = perturb_fn(image)

        fig, axs = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle(f"Question: {question}", fontsize=11)

        axs[0].imshow(image)
        axs[0].set_title("Original")
        axs[0].axis("off")
        axs[0].text(0, -10, f"GT: {answer}", fontsize=9, color='green')
        axs[0].text(0, -25, f"Pred: {preds_clean[int(idx)]}", fontsize=9, color='blue')

        axs[1].imshow(perturbed_image)
        axs[1].set_title("Perturbed")
        axs[1].axis("off")
        axs[1].text(0, -10, f"Pred: {preds_perturbed[int(idx)]}", fontsize=9, color='red')

        plt.tight_layout()
        plt.show()


In [ ]:
# Example using Gaussian noise visualization
visualize_perturbed_samples(
    data=data['test'],
    preds_clean=preds,
    preds_perturbed=image_attacked_preds,
    perturb_fn=add_gaussian_noise,
    num_samples=5
)

In [ ]:
# Example using Gaussian noise visualization
visualize_perturbed_samples(
    data=data['test'],
    preds_clean=preds,
    preds_perturbed=image_attacked_preds,
    perturb_fn=jpeg_compress,
    num_samples=5
)

## PGD and FGSM

In [ ]:
model.eval()

def clean_answer(text):
    if "Assistant:" in text:
        return text.split("Assistant:")[-1].strip()
    return text.strip()



_img_proc = processor.image_processor
_MEAN = torch.tensor(_img_proc.image_mean).view(1, 1, 3, 1, 1)
_STD  = torch.tensor(_img_proc.image_std).view(1, 1, 3, 1, 1)

def _to_unit(pv):

    return pv * _STD.to(pv.device, pv.dtype) + _MEAN.to(pv.device, pv.dtype)

def _to_norm(unit):

    return (unit - _MEAN.to(unit.device, unit.dtype)) / _STD.to(unit.device, unit.dtype)


_IMAGE_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids("<image>")


if _IMAGE_TOKEN_ID is None or _IMAGE_TOKEN_ID == processor.tokenizer.unk_token_id:
    vocab = processor.tokenizer.get_vocab()
    if "<image>" in vocab:
        _IMAGE_TOKEN_ID = vocab["<image>"]
    else:

        _IMAGE_TOKEN_ID = getattr(processor, "image_token_id", None) \
            or getattr(model.config, "image_token_id", None) \
            or getattr(getattr(model.config, "text_config", None), "image_token_id", None)


print("image_token_id:", _IMAGE_TOKEN_ID)



image_token_id: 128257


In [ ]:
def build_supervised_inputs(image, question, answer):

    ans = answer if isinstance(answer, str) else answer[0]
    messages = [
        {"role": "user", "content": [
            {"type": "text", "text": "Answer as a medical specialist"},
            {"type": "image"},
            {"type": "text", "text": question},
        ]},
        {"role": "assistant", "content": [{"type": "text", "text": ans}]},
    ]
    text = processor.apply_chat_template(messages, add_generation_prompt=False)
    batch = processor(text=text.strip(), images=[[image]], return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == _IMAGE_TOKEN_ID] = -100
    batch["labels"] = labels
    return {k: v.to(model.device) for k, v in batch.items()}


def build_generation_inputs(image, question):

    messages = [{"role": "user", "content": [
        {"type": "text", "text": "Answer as a medical specialist"},
        {"type": "image"},
        {"type": "text", "text": question},
    ]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[[image]], return_tensors="pt", padding=True)
    return {k: v.to(model.device) for k, v in inputs.items()}


In [ ]:
def pgd_on_pixels(inputs, eps=8/255, alpha=2/255, num_iter=10, random_start=True):

    model.eval()
    pv = inputs["pixel_values"].detach()
    unit_clean = _to_unit(pv).detach()
    other = {k: v for k, v in inputs.items() if k != "pixel_values"}

    delta = (torch.empty_like(unit_clean).uniform_(-eps, eps)
             if random_start else torch.zeros_like(unit_clean)).to(unit_clean.dtype)

    for _ in range(num_iter):
        delta.requires_grad_(True)
        unit_adv = torch.clamp(unit_clean + delta, 0.0, 1.0)
        out = model(pixel_values=_to_norm(unit_adv), **other)
        grad = torch.autograd.grad(out.loss, delta)[0]
        with torch.no_grad():
            delta = delta + alpha * grad.sign()               
            delta = torch.clamp(delta, -eps, eps)
            delta = torch.clamp(unit_clean + delta, 0.0, 1.0) - unit_clean
        delta = delta.detach()

    unit_adv = torch.clamp(unit_clean + delta, 0.0, 1.0).detach()
    return _to_norm(unit_adv).detach()


def fgsm_on_pixels(inputs, eps=8/255):
    return pgd_on_pixels(inputs, eps=eps, alpha=eps, num_iter=1, random_start=False)


In [ ]:
def _find_vision_feature_module():
    cands = []
    for name, _ in model.named_modules():
        low = name.lower()
        if low.endswith("connector") or "modality_projection" in low or "perceiver_resampler" in low:
            cands.append(name)
    if not cands:
        for name, _ in model.named_modules():
            if name.lower().endswith("vision_model"):
                cands.append(name)
    cands.sort(key=len, reverse=True)
    return cands[0] if cands else None

_FEATURE_MODULE_NAME = _find_vision_feature_module()
print("Vision-feature hook target:", _FEATURE_MODULE_NAME)

def _get_module_by_name(name):
    mod = model
    for p in name.split("."):
        mod = getattr(mod, p)
    return mod

def vision_encoder_attack(image, question, eps=8/255, alpha=2/255,
                          num_iter=10, random_start=True):

    assert _FEATURE_MODULE_NAME is not None, "No vision feature module found"
    model.eval()
    inputs = build_generation_inputs(image, question)
    pv = inputs["pixel_values"].detach()
    unit_clean = _to_unit(pv).detach()
    other = {k: v for k, v in inputs.items() if k != "pixel_values"}

    target_mod = _get_module_by_name(_FEATURE_MODULE_NAME)
    captured = {}
    def hook(_m, _i, output):
        captured["feat"] = output[0] if isinstance(output, tuple) else output
    handle = target_mod.register_forward_hook(hook)

    try:
        with torch.no_grad():
            model(pixel_values=pv, **other)
            feat_clean = captured["feat"].detach().float()

        delta = (torch.empty_like(unit_clean).uniform_(-eps, eps)
                 if random_start else torch.zeros_like(unit_clean)).to(unit_clean.dtype)

        for _ in range(num_iter):
            delta.requires_grad_(True)
            unit_adv = torch.clamp(unit_clean + delta, 0.0, 1.0)
            model(pixel_values=_to_norm(unit_adv), **other)
            loss = F.mse_loss(captured["feat"].float(), feat_clean)  
            grad = torch.autograd.grad(loss, delta)[0]
            with torch.no_grad():
                delta = delta + alpha * grad.sign()
                delta = torch.clamp(delta, -eps, eps)
                delta = torch.clamp(unit_clean + delta, 0.0, 1.0) - unit_clean
            delta = delta.detach()
    finally:
        handle.remove()

    unit_adv = torch.clamp(unit_clean + delta, 0.0, 1.0).detach()
    return _to_norm(unit_adv).detach()


In [ ]:
_CROSSMODAL_SYNONYMS = {
    "image": ["picture", "scan", "frame"], "polyp": ["lesion", "growth", "mass"],
    "finding": ["observation", "result"], "abnormalities": ["irregularities", "issues"],
    "present": ["visible", "shown"], "size": ["dimension", "extent"],
    "color": ["colour", "shade"], "where": ["in which region", "at what site"],
    "type": ["kind", "category"], "detect": ["identify", "spot"],
}

@torch.no_grad()
def _answer_loss(image, question, answer):
    return model(**build_supervised_inputs(image, question, answer)).loss.item()

def greedy_text_attack(image, question, answer, max_changes=3):

    words = question.split()
    best_q = question
    best_loss = _answer_loss(image, question, answer)
    changes = 0
    for i in range(len(words)):
        if changes >= max_changes:
            break
        w = words[i].lower().strip("?.,")
        cands = []
        if w in _CROSSMODAL_SYNONYMS:
            for syn in _CROSSMODAL_SYNONYMS[w]:
                c = words.copy(); c[i] = syn; cands.append(" ".join(c))
        if len(words) > 3:
            cands.append(" ".join(words[:i] + words[i+1:]))
        improved = False
        for cq in cands:
            l = _answer_loss(image, cq, answer)
            if l > best_loss:
                best_loss, best_q, improved = l, cq, True
        if improved:
            words = best_q.split(); changes += 1
    return best_q

def cross_modal_attack(image, question, answer, eps=8/255, alpha=2/255, num_iter=10):

    adv_q = greedy_text_attack(image, question, answer)
    inputs = build_supervised_inputs(image, adv_q, answer)
    adv_pv = pgd_on_pixels(inputs, eps=eps, alpha=alpha, num_iter=num_iter)
    return adv_pv, adv_q


@torch.no_grad()
def generate_from_pv(adv_pixel_values, question, image_for_template):
    gen = build_generation_inputs(image_for_template, question)
    gen["pixel_values"] = adv_pixel_values.to(gen["pixel_values"].dtype)
    out_ids = model.generate(**gen, max_new_tokens=64)
    return clean_answer(processor.batch_decode(out_ids, skip_special_tokens=True)[0])

def run_attack_eval(attack="pgd", n=None, eps=8/255, alpha=2/255, num_iter=10):
    test = data["test"]
    if n is not None:
        test = test.select(range(min(n, len(test))))

    preds, refs = [], []
    for ex in tqdm(test, desc=f"attack={attack}"):
        image, question, answer = ex["image"], ex["question"], ex["answer"]
        ans_str = answer[0] if isinstance(answer, list) else answer

        if attack == "clean":
            gen = build_generation_inputs(image, question)
            with torch.no_grad():
                out_ids = model.generate(**gen, max_new_tokens=64)
            pred = clean_answer(processor.batch_decode(out_ids, skip_special_tokens=True)[0])

        elif attack in ("pgd", "fgsm"):
            inp = build_supervised_inputs(image, question, ans_str)
            adv_pv = (fgsm_on_pixels(inp, eps=eps) if attack == "fgsm"
                      else pgd_on_pixels(inp, eps=eps, alpha=alpha, num_iter=num_iter))
            pred = generate_from_pv(adv_pv, question, image)

        elif attack == "vision":
            adv_pv = vision_encoder_attack(image, question, eps=eps, alpha=alpha, num_iter=num_iter)
            pred = generate_from_pv(adv_pv, question, image)

        elif attack == "crossmodal":
            adv_pv, adv_q = cross_modal_attack(image, question, ans_str, eps=eps, alpha=alpha, num_iter=num_iter)
            pred = generate_from_pv(adv_pv, adv_q, image)
        else:
            raise ValueError(attack)

        preds.append(pred)
        refs.append(answer if isinstance(answer, list) else [answer])
    return preds, refs


In [ ]:
import evaluate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import Levenshtein

_bleu, _rouge, _meteor = evaluate.load("bleu"), evaluate.load("rouge"), evaluate.load("meteor")

def compute_metrics(preds, refs):
    refs_single = [r[0] for r in refs]
    accuracy = sum(p in r for p, r in zip(preds, refs)) / len(refs) * 100
    bleu = _bleu.compute(predictions=preds, references=[[r] for r in refs_single])["bleu"]
    rouge = _rouge.compute(predictions=preds, references=refs_single)["rougeL"]
    meteor = _meteor.compute(predictions=preds, references=refs_single)["meteor"]
    j = []
    for r, p in zip(refs_single, preds):
        sr, sp = set(r.split()), set(p.split())
        j.append(len(sr & sp) / len(sr | sp) if (sr | sp) else 0)
    jaccard = sum(j) / len(j) * 100
    vec = TfidfVectorizer().fit(refs_single + preds)
    cosine = cosine_similarity(vec.transform(refs_single), vec.transform(preds)).diagonal().mean() * 100
    return {"accuracy": round(accuracy, 2), "bleu": round(bleu, 4),
            "rougeL": round(rouge, 4), "meteor": round(meteor, 4),
            "jaccard": round(jaccard, 2), "cosine": round(cosine, 2)}


[nltk_data] Downloading package wordnet to /home/rifat/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/rifat/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/rifat/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
def attack_success_rate(clean_preds, adv_preds):
    changed = sum(c.strip() != a.strip() for c, a in zip(clean_preds, adv_preds))
    return changed / len(clean_preds) * 100

In [ ]:
def is_correct(pred, refs):

    return pred in refs

def run_all_attacks_full(attacks=("clean", "fgsm", "pgd"),
                         n=None, eps=8/255, alpha=2/255, num_iter=5):

    store = {}
    refs_ref = None
    for atk in attacks:
        preds, refs = run_attack_eval(attack=atk, n=n, eps=eps, alpha=alpha, num_iter=num_iter)
        store[atk] = preds
        refs_ref = refs  
        print(f"[done] {atk}: {len(preds)} preds")
    return store, refs_ref



In [ ]:

preds_store, refs = run_all_attacks_full(attacks=("fgsm", "pgd"), n=None, eps=8/255, alpha=2/255, num_iter=10)

preds_store["clean"] = preds
refs = refs

attack=fgsm: 100%|██████████| 6355/6355 [6:16:19<00:00,  3.55s/it]   


[done] fgsm: 6355 preds


attack=pgd: 100%|██████████| 6355/6355 [22:15:56<00:00, 12.61s/it]   

[done] pgd: 6355 preds


In [ ]:
def attack_success_rate_changed(clean_preds, adv_preds): # this is considered in this study

    changed = sum(c.strip() != a.strip() for c, a in zip(clean_preds, adv_preds))
    return changed / len(clean_preds) * 100

def attack_success_rate_flip(clean_preds, adv_preds, refs):

    flipped, base = 0, 0
    for c, a, r in zip(clean_preds, adv_preds, refs):
        if is_correct(c, r):              # only count examples the model got right when clean
            base += 1
            if not is_correct(a, r):
                flipped += 1
    return (flipped / base * 100) if base else 0.0

clean_preds = preds_store["clean"]

print(f"{'attack':<12}{'accuracy':>10}{'ASR_changed':>13}{'ASR_flip':>10}")
print("-" * 45)
summary = {}
for atk, preds in preds_store.items():
    m = compute_metrics(preds, refs)
    asr_changed = attack_success_rate_changed(clean_preds, preds) if atk != "clean" else 0.0
    asr_flip    = attack_success_rate_flip(clean_preds, preds, refs) if atk != "clean" else 0.0
    summary[atk] = {**m, "ASR_changed": round(asr_changed, 2), "ASR_flip": round(asr_flip, 2)}
    print(f"{atk:<12}{m['accuracy']:>10}{asr_changed:>13.2f}{asr_flip:>10.2f}")

attack        accuracy  ASR_changed  ASR_flip
---------------------------------------------
fgsm             67.22        29.24     22.89
pgd              65.57        30.57     24.62
clean            85.52         0.00      0.00


In [ ]:

metrics = ["accuracy", "bleu", "rougeL", "meteor", "jaccard", "cosine"]
header = f"{'attack':<12}" + "".join(f"{m:>10}" for m in metrics)
print(header); print("-" * len(header))
for atk in preds_store:
    r = summary[atk]
    print(f"{atk:<12}" + "".join(f"{r[m]:>10}" for m in metrics))

attack        accuracy      bleu    rougeL    meteor   jaccard    cosine
------------------------------------------------------------------------
fgsm             67.22    0.6771    0.7042    0.4128     69.01     60.22
pgd              65.57      0.67    0.6877    0.4019     67.34     58.63
clean            85.52    0.7584    0.9047     0.519     88.24     75.99
